In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Current working directory:", Path.cwd())

# Try both possible locations depending on where the kernel's cwd actually is
candidates = [
    Path("data/rainfall_dataset.xls"),       # if cwd = droplet_ai/ (workspace root)
    Path("../data/rainfall_dataset.xls"),    # if cwd = notebooks/
]

DATA_PATH = None
for candidate in candidates:
    if candidate.exists():
        DATA_PATH = candidate
        break

if DATA_PATH is None:
    print("File not found in either expected location. Contents of cwd:")
    print(list(Path.cwd().iterdir()))
    raise FileNotFoundError("rainfall_dataset.xls not found — see directory listing above")

print("Found file at:", DATA_PATH.resolve())

df_raw = pd.read_excel(DATA_PATH, engine="xlrd", header=None)
print("Raw shape:", df_raw.shape)
df_raw.head(6)

Current working directory: c:\Users\USER\Desktop\droplet_ai\notebooks
Found file at: C:\Users\USER\Desktop\droplet_ai\data\rainfall_dataset.xls
Raw shape: (649, 35)


,0,1,2,3,4,5,6,7,8,9,...,25,26,27,28,29,30,31,32,33,34
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Name,Year,Month,Time,DAY1,DAY2,DAY3,DAY4,DAY5,DAY6,...,DAY22,DAY23,DAY24,DAY25,DAY26,DAY27,DAY28,DAY29,DAY30,DAY31
4,Takoradi,1996,01,09:00,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,Takoradi,1996,02,09:00,0,0,5.2,0,0,0,...,0.3,0,0,0,0,0,0,0,NaN,NaN


In [4]:
# Apply the real header row (index 3) and keep only the data rows below it
df = df_raw.iloc[4:].copy()
df.columns = df_raw.iloc[3]
df = df.reset_index(drop=True)

# Drop fully blank separator rows
df = df.dropna(how="all")

# Drop the embedded duplicate header row (where Name == "Name")
df = df[df["Name"] != "Name"]

# Keep Tarkwa-U.M.A.T. only — drop Takoradi entirely
df = df[df["Name"] == "Tarkwa-U.M.A.T."].reset_index(drop=True)

print("Tarkwa-only shape:", df.shape)
print("Unique station names remaining:", df["Name"].unique())
print("Year range:", df["Year"].min(), "to", df["Year"].max())
df.head()

Tarkwa-only shape: (285, 35)
Unique station names remaining: <StringArray>
['Tarkwa-U.M.A.T.']
Length: 1, dtype: str
Year range: 1996 to 2019


3,Name,Year,Month,Time,DAY1,DAY2,DAY3,DAY4,DAY5,DAY6,...,DAY22,DAY23,DAY24,DAY25,DAY26,DAY27,DAY28,DAY29,DAY30,DAY31
0,Tarkwa-U.M.A.T.,1996,01,09:00,0,0,0,0,0,3.6,...,0,0,0,0,1,0,0,0,0,0
1,Tarkwa-U.M.A.T.,1996,02,09:00,0,0,0,0,0,0,...,0,0,0,0,1.1,0,1.2,0,NaN,NaN
2,Tarkwa-U.M.A.T.,1996,03,09:00,5.6,1.5,0,0,1.7,0,...,0,0,0,9.4,0,0,7.3,0,0,0
3,Tarkwa-U.M.A.T.,1996,04,09:00,0,0,0,0,10.4,24.4,...,2.8,1.2,3.2,2.5,0,8.2,0,79.2,0,NaN
4,Tarkwa-U.M.A.T.,1996,05,09:00,0,0,0,5.4,74.5,0,...,0,1.6,3.3,0.3,0,38,1.6,1.1,90,0


In [5]:
# Convert Year to integer
df["Year"] = df["Year"].astype(int)

# Keep a numeric month for sorting and seasonal modelling (SARIMA needs this internally)
df["Month_Num"] = df["Month"].astype(int)

# Map numeric month to full month name — used everywhere user-facing
month_names = {
    1: "January", 2: "February", 3: "March", 4: "April",
    5: "May", 6: "June", 7: "July", 8: "August",
    9: "September", 10: "October", 11: "November", 12: "December"
}
df["Month_Name"] = df["Month_Num"].map(month_names)

# Drop the old raw Month and Time columns now that we have clean replacements
df = df.drop(columns=["Month", "Time"])

# Sort chronologically — essential before we reshape into a time series next
df = df.sort_values(["Year", "Month_Num"]).reset_index(drop=True)

print(df[["Name", "Year", "Month_Num", "Month_Name"]].head(15))
print("\nDtypes:")
print(df[["Year", "Month_Num", "Month_Name"]].dtypes)

3              Name  Year  Month_Num Month_Name
0   Tarkwa-U.M.A.T.  1996          1    January
1   Tarkwa-U.M.A.T.  1996          2   February
2   Tarkwa-U.M.A.T.  1996          3      March
3   Tarkwa-U.M.A.T.  1996          4      April
4   Tarkwa-U.M.A.T.  1996          5        May
5   Tarkwa-U.M.A.T.  1996          6       June
6   Tarkwa-U.M.A.T.  1996          7       July
7   Tarkwa-U.M.A.T.  1996          8     August
8   Tarkwa-U.M.A.T.  1996          9  September
9   Tarkwa-U.M.A.T.  1996         10    October
10  Tarkwa-U.M.A.T.  1996         11   November
11  Tarkwa-U.M.A.T.  1996         12   December
12  Tarkwa-U.M.A.T.  1997          1    January
13  Tarkwa-U.M.A.T.  1997          2   February
14  Tarkwa-U.M.A.T.  1997          3      March

Dtypes:
3
Year          int64
Month_Num     int64
Month_Name      str
dtype: object


In [6]:
day_cols = [f"DAY{i}" for i in range(1, 32)]

# Convert each day column to numeric; anything unexpected becomes NaN (we'll check for this)
for col in day_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Dtypes after conversion:")
print(df[day_cols].dtypes.unique())

print("\nMissing values per day column:")
print(df[day_cols].isna().sum())

print("\nValue range check:")
print("Min:", df[day_cols].min().min())
print("Max:", df[day_cols].max().max())
print("Any negative values?", (df[day_cols] < 0).any().any())

Dtypes after conversion:
[dtype('float64')]

Missing values per day column:
3
DAY1       0
DAY2       0
DAY3       0
DAY4       0
DAY5       0
DAY6       0
DAY7       0
DAY8       0
DAY9       0
DAY10      0
DAY11      0
DAY12      0
DAY13      0
DAY14      0
DAY15      0
DAY16      0
DAY17      0
DAY18      0
DAY19      0
DAY20      0
DAY21      0
DAY22      0
DAY23      0
DAY24      0
DAY25      0
DAY26      0
DAY27      0
DAY28      0
DAY29     18
DAY30     24
DAY31    119
dtype: int64

Value range check:
Min: 0.0
Max: 164.1
Any negative values? False


In [7]:
print("Total missing across all day columns:", df[day_cols].isna().sum().sum())
print("\nMissing by column (DAY21-31 only):")
print(df[day_cols[20:]].isna().sum())

Total missing across all day columns: 161

Missing by column (DAY21-31 only):
3
DAY21      0
DAY22      0
DAY23      0
DAY24      0
DAY25      0
DAY26      0
DAY27      0
DAY28      0
DAY29     18
DAY30     24
DAY31    119
dtype: int64


In [8]:
# Sum the real days in each month — skipna=True correctly ignores the
# calendar-structural NaNs (e.g. April has no DAY31) without needing interpolation
df["Monthly_Rainfall_mm"] = df[day_cols].sum(axis=1, skipna=True)

# Build a proper date column (first of each month) for time series indexing
df["Date"] = pd.to_datetime(dict(year=df["Year"], month=df["Month_Num"], day=1))

monthly_df = df[["Date", "Year", "Month_Num", "Month_Name", "Monthly_Rainfall_mm"]].copy()
monthly_df = monthly_df.sort_values("Date").reset_index(drop=True)

print("Shape:", monthly_df.shape)
print("\nFirst 5 rows:")
print(monthly_df.head())
print("\nLast 5 rows:")
print(monthly_df.tail())
print("\nSummary stats:")
print(monthly_df["Monthly_Rainfall_mm"].describe().round(2))

Shape: (285, 5)

First 5 rows:
3       Date  Year  Month_Num Month_Name  Monthly_Rainfall_mm
0 1996-01-01  1996          1    January                 10.8
1 1996-02-01  1996          2   February                 96.5
2 1996-03-01  1996          3      March                106.4
3 1996-04-01  1996          4      April                242.3
4 1996-05-01  1996          5        May                277.7

Last 5 rows:
3         Date  Year  Month_Num Month_Name  Monthly_Rainfall_mm
280 2019-05-01  2019          5        May                258.5
281 2019-06-01  2019          6       June                316.2
282 2019-07-01  2019          7       July                 65.2
283 2019-08-01  2019          8     August                 84.5
284 2019-09-01  2019          9  September                330.8

Summary stats:
count    285.0
mean     150.5
std      105.3
min        0.6
25%       65.6
50%      129.8
75%      213.0
max      525.9
Name: Monthly_Rainfall_mm, dtype: float64


In [9]:
OUTPUT_PATH = "../data/tarkwa_monthly_clean.csv"
monthly_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(monthly_df)} rows to {OUTPUT_PATH}")

# Quick reload check — confirms the file saved correctly and types survive the round-trip
check = pd.read_csv(OUTPUT_PATH, parse_dates=["Date"])
print("\nReloaded shape:", check.shape)
print("Reloaded dtypes:")
print(check.dtypes)

Saved 285 rows to ../data/tarkwa_monthly_clean.csv

Reloaded shape: (285, 5)
Reloaded dtypes:
Date                   datetime64[us]
Year                            int64
Month_Num                       int64
Month_Name                        str
Monthly_Rainfall_mm           float64
dtype: object
